# getitem-back-add-at — worked example 1: Backward of 1-D indexing via index_add_

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The derivative of `out = x[idx]` scatters `grad_out` into a zero tensor shaped like `x`, accumulating at the indexed positions. `index_add_` is the right primitive because repeated indices must SUM their contributions — a plain `grad_in[idx] = grad_out` would let the last write win and drop gradient.

## Worked solution

We compute the gradient of a 1-D gather `out = x[idx]`.

1. From `out[i] = x[idx[i]]`, the local derivative is 1 where `j == idx[i]` and 0 elsewhere. By the chain rule, `dL/dx[j] = sum over i with idx[i]==j of grad_out[i]`.
2. Allocate `grad_in = t.zeros_like(x)` — the gradient must have the shape of `x`, not of `out`.
3. Call `grad_in.index_add_(0, idx, grad_out)`. Along axis 0 it adds `grad_out[i]` into row `idx[i]`. If an index repeats, both contributions accumulate into the same slot — exactly the summation the chain rule requires.
4. We build an `idx` with a deliberate repeat and confirm the repeated position received the sum of the corresponding `grad_out` entries.

In [ ]:
import torch as t

t.manual_seed(0)
x = t.zeros(5)
idx = t.tensor([0, 2, 2, 4])
grad_out = t.tensor([1.0, 3.0, 5.0, 7.0])

def getitem_back(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    grad_in.index_add_(0, idx, grad_out)
    return grad_in

grad_in = getitem_back(grad_out, x, idx)
print(grad_in.tolist())
print('repeated index 2 summed:', float(grad_in[2]) == 8.0)